>Test cấu trúc

In [2]:
import json

FILE_PATH = "generations.jsonl"

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        obj = json.loads(line)
        print(obj.keys())
        # print(obj)

dict_keys(['run_id', 'question_idx', 'question', 'hint', 'label', 'model_output', 'output_token_length', 'finish_reason'])
dict_keys(['run_id', 'question_idx', 'question', 'hint', 'label', 'model_output', 'output_token_length', 'finish_reason'])
dict_keys(['run_id', 'question_idx', 'question', 'hint', 'label', 'model_output', 'output_token_length', 'finish_reason'])
dict_keys(['run_id', 'question_idx', 'question', 'hint', 'label', 'model_output', 'output_token_length', 'finish_reason'])
dict_keys(['run_id', 'question_idx', 'question', 'hint', 'label', 'model_output', 'output_token_length', 'finish_reason'])


In [3]:
import json

FILE_PATH = "generations.jsonl"

def split_thinking(text):
    if "</think>" in text:
        think, rest = text.split("</think>", 1)
        think = think.replace("<think>", "").strip()
        return think, rest.strip()
    return "", text

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break

        obj = json.loads(line)

        thinking, answer = split_thinking(obj["model_output"])

        print("\n" + "="*80)
        print(f"QUESTION_IDX: {obj['question_idx']}")
        print("-"*80)

        print("THINKING:")
        print(thinking)   # cắt 300 ký tự cho gọn

        print("-"*80)

        print("FINAL:")
        print(answer)


QUESTION_IDX: 00077f9cf9
--------------------------------------------------------------------------------
THINKING:
Okay, let's see. The problem says that the measures of a pair of supplementary angles are in the ratio of 7:2. I need to find the positive difference between their measures. Hmm, let me start by recalling what supplementary angles are. Supplementary angles are two angles that add up to 180 degrees. So, if they are in a ratio of 7:2, that means one angle is 7 parts and the other is 2 parts. 

The hint says to let the supplementary angles be 7x and 2x. That makes sense because the ratio is 7:2. So, if I let x be a variable, then the two angles would be 7x and 2x. 

Now, since they are supplementary, their sum should be 180 degrees. So, the equation would be 7x + 2x = 180. Let me check that. Adding 7x and 2x gives 9x, which equals 180. So, solving for x, I divide both sides by 9. That would give x = 180 / 9. Let me calculate that. 180 divided by 9 is 20. So, x is 20. 

Wait

>Thống kê reason_stop

In [1]:
import json
from collections import Counter

FILE_PATH = "generations.jsonl"

counter = Counter()

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        finish = obj.get("finish_reason", "unknown")
        counter[finish] += 1

print("Finish reason stats:")
for k, v in counter.items():
    print(f"{k}: {v}")

Finish reason stats:
stop: 6433
length: 865


>Số lượng output > 4000

In [6]:
import json

FILE_PATH = "generations.jsonl"

count = 0
total = 0

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        total += 1

        if obj.get("output_token_length", 0) > 4000:
            count += 1

print(f">4000: {count}/{total} ({count/total*100:.2f}%)")

>4000: 0/7298 (0.00%)


>Chiều dài trung bình khi `stop`, chứng tỏ có những trường hợp đề bài quá dài

In [7]:
import json

FILE_PATH = "generations.jsonl"

length_tokens = []

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        if obj.get("finish_reason") == "length":
            length_tokens.append(obj.get("output_token_length", 0))

# stats
if length_tokens:
    print(f"Total length samples: {len(length_tokens)}")
    print(f"Min: {min(length_tokens)}")
    print(f"Max: {max(length_tokens)}")
    print(f"Avg: {sum(length_tokens)/len(length_tokens):.2f}")
else:
    print("No length samples found")

Total length samples: 1042
Min: 3498
Max: 3987
Avg: 3853.17


>Debug, kiểm tra các câu có input dài bất thường

In [8]:
import json

FILE_PATH = "generations.jsonl"

length_samples = []

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        if obj.get("finish_reason") == "length":
            length_samples.append({
                "question_idx": obj["question_idx"],
                "output_token_length": obj.get("output_token_length", 0),
                "question_hint": obj["full_text"],
            })

# sort tăng dần theo output_token_length
length_samples.sort(key=lambda x: x["output_token_length"])

# lấy top 10 nhỏ nhất
top10 = length_samples[:10]

print("\n===== TOP 10 SMALLEST LENGTH-STOP SAMPLES =====")

for i, sample in enumerate(top10):
    print("\n" + "-"*80)
    print(f"Rank {i+1}")
    print(f"QUESTION_IDX: {sample['question_idx']}")
    print(f"TOKENS      : {sample['output_token_length']}")
    print("QUESTION_HINT:")
    print(sample["question_hint"])  


KeyError: 'full_text'

>Chiều dài trung bình khi `length`, chứng tỏ có trường hợp bị out suy nghĩ

In [4]:
import json

FILE_PATH = "generations.jsonl"

length_tokens = []

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        if obj.get("finish_reason") == "stop":
            length_tokens.append(obj.get("output_token_length", 0))

# stats
if length_tokens:
    print(f"Total length samples: {len(length_tokens)}")
    print(f"Min: {min(length_tokens)}")
    print(f"Max: {max(length_tokens)}")
    print(f"Avg: {sum(length_tokens)/len(length_tokens):.2f}")
else:
    print("No length samples found")

Total length samples: 6380
Min: 9
Max: 8057
Avg: 1765.85
